In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv('../data/movies.csv')
ratings = pd.read_csv('../data/ratings.csv')

In [2]:
#Prepare Genre Data 
# Replace pipe separator with space so TF-IDF can read each genre as a word
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

print(movies[['title', 'genres', 'genres_clean']].head())

                                title  \
0                    Toy Story (1995)   
1                      Jumanji (1995)   
2             Grumpier Old Men (1995)   
3            Waiting to Exhale (1995)   
4  Father of the Bride Part II (1995)   

                                        genres  \
0  Adventure|Animation|Children|Comedy|Fantasy   
1                   Adventure|Children|Fantasy   
2                               Comedy|Romance   
3                         Comedy|Drama|Romance   
4                                       Comedy   

                                  genres_clean  
0  Adventure Animation Children Comedy Fantasy  
1                   Adventure Children Fantasy  
2                               Comedy Romance  
3                         Comedy Drama Romance  
4                                       Comedy  


In [3]:
#Build TF-IDF Matrix
# TF-IDF converts text into numbers
# Each movie becomes a vector of genre weights
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)
# Shape will be (9742 movies, number of unique genres)

TF-IDF Matrix Shape: (9742, 23)


In [4]:
# Calculate similarity between every pair of movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine Similarity Matrix Shape:", cosine_sim.shape)
# 9742 x 9742 matrix — every movie vs every other movie

Cosine Similarity Matrix Shape: (9742, 9742)


In [5]:
#Build Index for Movie Lookup
# Create a mapping of movie title to index
# This lets us find a movie's row number quickly
indices = pd.Series(movies.index, index=movies['title']).drop_duplicates()

print("Sample indices:")
print(indices.head())

Sample indices:
title
Toy Story (1995)                      0
Jumanji (1995)                        1
Grumpier Old Men (1995)               2
Waiting to Exhale (1995)              3
Father of the Bride Part II (1995)    4
dtype: int64


In [6]:
#The Recommendation Function

def get_content_recommendations(title, top_n=10):
    """
    Given a movie title, return top N similar movies based on genre.
    """
    # Check if movie exists
    if title not in indices:
        return f"Movie '{title}' not found in dataset."
    
    # Get index of the movie
    idx = indices[title]
    
    # Get similarity scores for this movie with all others
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity score (descending)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Skip the first one (it's the movie itself, similarity = 1.0)
    sim_scores = sim_scores[1:top_n+1]
    
    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]
    similarity_scores = [round(i[1], 3) for i in sim_scores]
    
    # Return results
    result = movies.iloc[movie_indices][['title', 'genres']].copy()
    result['similarity_score'] = similarity_scores
    
    return result

# Test it
print("=== Movies similar to Toy Story (1995) ===")
print(get_content_recommendations('Toy Story (1995)'))

=== Movies similar to Toy Story (1995) ===
                                                  title  \
1706                                        Antz (1998)   
2355                                 Toy Story 2 (1999)   
2809     Adventures of Rocky and Bullwinkle, The (2000)   
3000                   Emperor's New Groove, The (2000)   
3568                              Monsters, Inc. (2001)   
6194                                   Wild, The (2006)   
6486                             Shrek the Third (2007)   
6948                     Tale of Despereaux, The (2008)   
7760  Asterix and the Vikings (Astérix et les Viking...   
8219                                       Turbo (2013)   

                                           genres  similarity_score  
1706  Adventure|Animation|Children|Comedy|Fantasy               1.0  
2355  Adventure|Animation|Children|Comedy|Fantasy               1.0  
2809  Adventure|Animation|Children|Comedy|Fantasy               1.0  
3000  Adventure|Animation|C

In [12]:
# Cell 6B Improved Recommendation Function with Tiebreaker 
# Load popular movies for tiebreaker
popular_movies = pd.read_csv('../data/popular_movies.csv')

def get_content_recommendations(title, top_n=10):
    """
    Given a movie title, return top N similar movies based on genre.
    Uses weighted score as tiebreaker when similarity scores are equal.
    """
    if title not in indices:
        return f"Movie '{title}' not found in dataset."
    
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    movie_indices = [i[0] for i in sim_scores]
    similarity_scores = [round(i[1], 3) for i in sim_scores]
    
    result = movies.iloc[movie_indices][['movieId', 'title', 'genres']].copy()
    result['similarity_score'] = similarity_scores
    
    # Merge with weighted score as tiebreaker
    result = result.merge(
        popular_movies[['movieId', 'weighted_score', 'rating_count']],
        on='movieId',
        how='left'
    )
    
    # Fill NaN weighted scores with 0 (movies with less than 50 ratings)
    result['weighted_score'] = result['weighted_score'].fillna(0)
    
    # Sort by similarity first, then by weighted score as tiebreaker
    result = result.sort_values(
        ['similarity_score', 'weighted_score'],
        ascending=[False, False]
    )
    
    return result[['title', 'genres', 'similarity_score', 'weighted_score', 'rating_count']].head(top_n)

# Test again
print("=== Movies similar to Toy Story (1995) ===")
print(get_content_recommendations('Toy Story (1995)'))

=== Movies similar to Toy Story (1995) ===
                                               title  \
4                              Monsters, Inc. (2001)   
1                                 Toy Story 2 (1999)   
0                                        Antz (1998)   
2     Adventures of Rocky and Bullwinkle, The (2000)   
3                   Emperor's New Groove, The (2000)   
5                                   Wild, The (2006)   
6                             Shrek the Third (2007)   
7                     Tale of Despereaux, The (2008)   
8  Asterix and the Vikings (Astérix et les Viking...   
9                                       Turbo (2013)   

                                        genres  similarity_score  \
4  Adventure|Animation|Children|Comedy|Fantasy               1.0   
1  Adventure|Animation|Children|Comedy|Fantasy               1.0   
0  Adventure|Animation|Children|Comedy|Fantasy               1.0   
2  Adventure|Animation|Children|Comedy|Fantasy               1.0   


In [13]:
#Test with more movies
print("=== Movies similar to The Matrix (1999) ===")
print(get_content_recommendations('Matrix, The (1999)'))

=== Movies similar to The Matrix (1999) ===
                    title                  genres  similarity_score  \
9      Matrix, The (1999)  Action|Sci-Fi|Thriller               1.0   
4     Blade Runner (1982)  Action|Sci-Fi|Thriller               1.0   
7  Terminator, The (1984)  Action|Sci-Fi|Thriller               1.0   
1  Johnny Mnemonic (1995)  Action|Sci-Fi|Thriller               1.0   
0        Screamers (1995)  Action|Sci-Fi|Thriller               1.0   
2       Virtuosity (1995)  Action|Sci-Fi|Thriller               1.0   
3          Timecop (1994)  Action|Sci-Fi|Thriller               1.0   
5             Solo (1996)  Action|Sci-Fi|Thriller               1.0   
6     Arrival, The (1996)  Action|Sci-Fi|Thriller               1.0   
8         Godzilla (1998)  Action|Sci-Fi|Thriller               1.0   

   weighted_score  rating_count  
9        4.087128         278.0  
4        3.928608         124.0  
7        3.787723         131.0  
1        3.078426          53.0  
0   

In [14]:
print("\n=== Movies similar to Schindler's List ===")
print(get_content_recommendations("Schindler's List (1993)"))


=== Movies similar to Schindler's List ===
                                         title     genres  similarity_score  \
3                      Schindler's List (1993)  Drama|War               1.0   
7                               Platoon (1986)  Drama|War               1.0   
0                       Misérables, Les (1995)  Drama|War               1.0   
1        Before the Rain (Pred dozhdot) (1994)  Drama|War               1.0   
2                     Walking Dead, The (1995)  Drama|War               1.0   
4  Land and Freedom (Tierra y libertad) (1995)  Drama|War               1.0   
5                            Stalingrad (1993)  Drama|War               1.0   
6                      Nothing Personal (1995)  Drama|War               1.0   
8     Tin Drum, The (Blechtrommel, Die) (1979)  Drama|War               1.0   
9                        Paths of Glory (1957)  Drama|War               1.0   

   weighted_score  rating_count  
3        4.091029         220.0  
7        3.770600 

In [15]:
#Save cosine similarity matrix

import pickle

# Save the cosine similarity matrix and indices for use in the app later
with open('../data/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

indices.to_csv('../data/movie_indices.csv', header=True)

print("Saved cosine similarity matrix and indices!")

Saved cosine similarity matrix and indices!
